#Lab: Lakehouse Concepts

##**Course 1, Week 1: Lakehouse Architecture & Platform**
 
###Objectives
- Identify the key properties of a data lakehouse
- Compare lakehouse vs data warehouse vs data lake
- Explore Delta Lake as the lakehouse storage layer
- Understand the Databricks platform architecture

###Part 1: Platform Exploration
###Verify your Databricks environment.
####Print the Spark version and verify the environment

In [0]:
print(f"Spark Version: {spark.version}")

###Part 2: Architecture Comparison
####Create a DataFrame comparing architectures



In [0]:
architectures = spark.createDataFrame(
    [
        ("Data Warehouse", True, True, False, False, True),
        ("Data Lake", False, False, True, True, False),
        ("Data Lakehouse", True, True, True, True, True)
    ],
    ["architecture", "acid_transactions", "schema_enforcement", "unstructured_data", "low_cost_storage", "bi_support"],
)

architectures.show(truncate=False)

###Part 3: Delta Lake Basics
####Create a simple Delta table with sample data and inspect its properties

In [0]:
# Step 1: Create sample data (at least 5 rows with columns: id, name, value)
sample_data = [
    (1, "Alice", 100.5),
    (2, "Bob", 250.0),
    (3, "Charlie", 175.25),
    (4, "Diana", 300.75),
    (5, "Eve", 425.0)
]
df = spark.createDataFrame(sample_data, ["id", "name", "value"])

# Step 2: Write as Delta table named "lab1_lakehouse"
df.write.format("delta").mode("overwrite").saveAsTable("lab1_lakehouse")

# Step 3: Query the table to verify
spark.sql("SELECT * FROM lab1_lakehouse").show()

### Part 4: Explore the Transaction Log
#### Look at the Delta transaction log to understand how ACID works.
#### View the history of your table

In [0]:
# View Transaction History (this works!)
print("=== Initial Transaction History ===")
spark.sql("DESCRIBE HISTORY lab1_lakehouse").show(truncate=False)

# Get table details including format and properties
print("\n=== Table Details ===")
spark.sql("DESCRIBE DETAIL lab1_lakehouse").show(truncate=False)

# Demonstrate ACID: Perform an UPDATE operation
print("\n=== Performing UPDATE (demonstrates Atomicity) ===")
spark.sql("UPDATE lab1_lakehouse SET value = value * 1.1 WHERE id < 3")
print("Update completed successfully - all changes committed atomically")

# View the updated history showing the new version
print("\n=== Updated Transaction History (shows new version) ===")
spark.sql("DESCRIBE HISTORY lab1_lakehouse").show(truncate=False)

# Query the updated data
print("\n=== Updated Data ===")
spark.sql("SELECT * FROM lab1_lakehouse ORDER BY id").show()

# Demonstrate Time Travel (Isolation)
print("\n=== Time Travel: Read version 0 ===")
spark.sql("SELECT * FROM lab1_lakehouse VERSION AS OF 0").show()

print("\n=== ACID Properties Demonstrated ===")
print("✓ Atomicity: UPDATE completed as single transaction")
print("✓ Consistency: Schema enforced, constraints maintained")
print("✓ Isolation: Can read previous versions via time travel")
print("✓ Durability: All changes logged in transaction log (see HISTORY)")

##Lab Validation and Cleanup

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Spark is running
    try:
        version = spark.version
        checks.append(("Spark environment", True))
    except Exception:
        checks.append(("Spark environment", False))

    # Check 2: Delta table exists
    try:
        df = spark.sql("SELECT * FROM lab1_lakehouse")
        checks.append(("Delta table created", df.count() >= 5))
    except Exception:
        checks.append(("Delta table created", False))

    # Check 3: Architecture comparison has 3 rows
    try:
        checks.append(("Architecture comparison", architectures.count() == 3))
    except Exception:
        checks.append(("Architecture comparison", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab1_lakehouse")
except Exception:
    pass